### Chart: Gas storage - international comparison

Idea:
- Figure 2, *Role of Gas Storage* [report](https://www.energy-uk.org.uk/fuelling-the-future/the-role-of-gas-storage-in-ensuring-energy-security/)

Sources:
- Gas in storage - https://agsi.gie.eu/#/
- Gas consumption - IEA data
    - Energy institue [2025 report](https://www.energyinst.org/statistical-review/resources-and-data-downloads)


Other:
- https://github.com/ccan23/iea_electricity_generation_data_scraper
- https://energiedashboard.admin.ch/gas/eu-gasspeicher
- https://www.consilium.europa.eu/en/infographics/gas-storage-capacity/



We can get live storage, and annual usage for all but UK. Need to get accurate final consumption of gas, to calculate storage days.

UK data sources:
- data.nationalgas
- or easier: https://terravolt.co.uk/uk-gas-visual/

Total natural gas storage: 35,415GWh

- COnsumption 61.9 Bcm. Standard IEA GCV factor of ~11.537. -> 714.1403 TWh


In [1]:
import pandas as pd
import country_converter as coco
import altair as alt
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name="article")

import pathlib
import os
from datetime import date

Set chart storage path

In [2]:
# What is my current working directory?
print(os.getcwd())

/Users/j.i.hellings/Documents/GitHub/RADataHub/ChartOfTheDay/gas-storage


**SET PUBLISH DATE FOR CHART** (used for chart name & folder)

In [14]:
# Find charts directory: 'ChartOfTheDay/charts'

save_date = date.today().strftime("%Y%m%d")
# save_date = '20260309' # OR MANUALLY SET IF NOT TODAY
save_month = date.today().strftime("%Y%m")


folder_path = f'../charts/{save_month}'
file_base = f'{save_date}_'

In [11]:
xmax=9
mo=0.5
logo=alt.Chart(pd.DataFrame([{"x": xmax, "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40,height=40,align='right',baseline='top',yOffset=-33,opacity=mo,xOffset=0).encode(x='x:Q',url='img:N')

logo

alt.Chart(...)

UK on 3rd-5th March:
    'Type': 'country',
    'Name': 'United Kingdom (Post-Brexit)',
    'Gas in storage (TWh)': 6.4639,
    'Full (%)': 6.4639 / 35.415 * 100,
    'Consumption (TWh)': 714.14,

    

In [5]:
gas_storage_raw = pd.read_csv('AGSI_gasDayStart_2026-03-07.csv', delimiter=';')
# 
gas_storage = gas_storage_raw[(gas_storage_raw['Type'] == 'country') | (gas_storage_raw['Type'] == 'EU')].copy()

# Augment UK values
uk_data = {
    'Type': 'country',
    'Name': 'United Kingdom (Post-Brexit)',
    'Gas in storage (TWh)': 7.25,
    'Full (%)': 7.25 / 35.415 * 100,
    'Consumption (TWh)': 714.14,
}
# Update values in last (UK) row with our augmented UK data, leaving the rest of the data unchanged
gas_storage.iloc[-1] = uk_data
gas_storage

,Type,Status,Name,Gas in storage (TWh),Full (%),Trend (%),Consumption (TWh),Stock/Cons (%),Injection (GWh/d),Withdrawal (GWh/d),Technical Capacity (TWh),Injection Capacity (GWh/d),Withdrawal Capacity (GWh/d),Contracted Capacity (TWh),Available Capacity (TWh),Dataset Coverage (%)
0,EU,E,EU,335.9200,29.400000,-0.02,3519.00,9.55,1545.05,1716.2,1142.6337,12173.88,20238.31,1033.9663,111.0672,100.0
1,country,C,Austria,36.2316,35.950000,0.00,78.20,46.33,44.86,44.5,100.7909,863.60,1083.51,101.3803,0.0078,100.0
12,country,C,Belgium,2.2964,25.570000,-0.23,152.95,1.50,0.00,8.4,8.9800,88.14,169.66,8.9812,0.0000,100.0
15,country,C,Bulgaria,2.8096,40.110000,-0.37,31.05,9.05,0.00,26.1,7.0044,42.13,46.88,5.1600,1.4400,100.0
18,country,C,Croatia,0.4960,10.390000,-0.03,27.60,1.80,4.65,5.8,4.7725,43.87,51.57,4.7725,0.0000,100.0
21,country,C,Czech Republic,14.3156,30.420000,-0.02,77.05,18.58,51.84,58.6,47.0535,558.17,732.07,47.3211,0.6418,100.0
30,country,C,Denmark,2.6408,26.970000,-0.21,18.40,14.35,5.29,25.5,9.7900,90.72,180.00,6.9965,1.9610,100.0
33,country,C,France,27.4830,21.860000,0.11,368.00,7.47,416.23,276.8,125.7231,1102.09,2500.74,125.7231,0.0000,100.0
44,country,E,Germany,53.1171,21.150000,0.19,903.90,5.88,743.54,260.6,251.1442,4258.00,7086.73,208.2001,44.0267,100.0
136,country,C,Hungary,23.1828,34.100000,-0.07,94.30,24.58,18.22,63.0,67.9909,509.32,798.90,67.4683,0.5225,100.0


In [6]:
gas_df = gas_storage[['Name', 'Gas in storage (TWh)', 'Full (%)', 'Consumption (TWh)']].copy()
gas_df.columns = ['country', 'stock', 'capacity_load', 'consumption']

# Add ISO3 country codes
gas_df['iso3'] = coco.convert(names=gas_df['country'], to='ISO3')
# re-order
gas_df = gas_df[['country', 'iso3', 'stock', 'capacity_load', 'consumption']].copy()

# Drop the Pre-Brexit UK row, which is now redundant, and non-eu (Serbia and Ukraine
gas_df = gas_df[~gas_df['country'].isin(['United Kingdom (Pre-Brexit)', 'Serbia', 'Ukraine'])].copy()
gas_df['country'] = gas_df['country'].str.replace({' (Post-Brexit)': ''})

# Calculate storage in days
gas_df['storage_days'] = gas_df['stock'] / (gas_df['consumption'] / 365)

# Calculate measure for size of maximum possible storage relative to consumption, as a measure of "capacity proficiency"
gas_df['storage_maximum'] = gas_df['stock'] * 100 / gas_df['capacity_load']
gas_df['capacity_proficiency'] = gas_df['storage_maximum'] / (gas_df['consumption'] / 365)

gas_df

EU not found in ISO2


,country,iso3,stock,capacity_load,consumption,storage_days,storage_maximum,capacity_proficiency
0,EU,not found,335.9200,29.400000,3519.00,34.842512,1142.585034,118.511946
1,Austria,AUT,36.2316,35.950000,78.20,169.111688,100.783310,470.408033
12,Belgium,BEL,2.2964,25.570000,152.95,5.480131,8.980837,21.431876
15,Bulgaria,BGR,2.8096,40.110000,31.05,33.027504,7.004737,82.342319
18,Croatia,HRV,0.4960,10.390000,27.60,6.559420,4.773821,63.132053
21,Czech Republic,CZE,14.3156,30.420000,77.05,67.815626,47.059829,222.931053
30,Denmark,DNK,2.6408,26.970000,18.40,52.385435,9.791620,194.235947
33,France,FRA,27.4830,21.860000,368.00,27.258954,125.722781,124.697867
44,Germany,DEU,53.1171,21.150000,903.90,21.448989,251.144681,101.413661
136,Hungary,HUN,23.1828,34.100000,94.30,89.731941,67.984751,263.143521


In [7]:
# Conditional colours
colour_lookup = {        'OECD': '#eb5c2e', # Orange
    'OECD': '#179fdb', # Orange
    'GBR': '#122b39',  # Dark blue
}

Figure 1. **Storage stock** TWh

In [8]:
len(gas_df)

21

In [9]:
gas_df.round(3).to_csv('gas_storage_2026-03-07.csv', index=False)

In [15]:
df_temp = gas_df[~gas_df['country'].isin(['EU', 'Ireland'])].copy()

bars = alt.Chart(df_temp).mark_bar(
    color = alt.expr("datum.country == 'United Kingdom' ? '#e6224b' : '#122b39'")
).encode(
    alt.Y('country:N').sort('-x'),
    alt.X('stock:Q').title('Gas storage stock, 3rd March 2026').axis(
        labelExpr='datum.value > 0 ? datum.value + " TWh" : datum.value',
        labelAlign='center',
        zindex=1,
        titleFontSize=12
    )
    # color=alt.condition(
    #     alt.datum.iso3 == 'GBR',
    #     alt.value('#e6224b'),  # Color for the UK bar
    #     alt.value('#122b39')   # Color for other bars
    # )
)

labels = bars.mark_text(
    align=alt.expr('datum.stock > 9 ? "right" : "left"'),
    color=alt.expr('datum.stock > 9 ? "white" : datum.iso3 == "GBR" ? "#e6224b" : "#122b39"'),
    dx=alt.expr('datum.stock > 9 ? -3 : 4'),
    baseline='middle',
    # text = alt.expr('datum.stock > 0 ? format(datum.stock, ".1f") + " TWh" : "None"')
    text = alt.expr('datum.stock == 0 ? "None" : datum.stock < 0.1 ? "<0.1" : datum.iso3 == "GBR" ? format(datum.stock, ".1f") + " TWh" : format(datum.stock, ".1f")')
)

logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(0),url='img:N')

chart = (bars + labels + logo).properties(
    width=350,
    height=400
)
chart.title = alt.Title(
    ["Source: AGSI (European stock); National Gas (UK); Energy Institute (consumption)", "Notes: Stocks as of 7-9th March 2026; consumption based on 2024 figure"],
    orient='bottom',
    fontStyle='italic',
    fontSize=10,
    color='#676A8680',  # domain colour with 50% opacity
    fontWeight='normal',
    frame='group',
    dx=0,
    offset=7
)

# pathlib.Path('') # use this to save in current directory
styles.save(chart, folder_path, f'{file_base}C1_gas_storage_stock_test', width=350, height=400)
chart.display()

alt.LayerChart(...)

In [ ]:
chart

**Capacity load**

In [240]:
gas_df.head()

,country,iso3,stock,capacity_load,consumption,storage_days,storage_maximum,capacity_proficiency
0,EU,not found,335.9200,29.40,3519.00,34.842512,1142.585034,118.511946
1,Austria,AUT,36.2316,35.95,78.20,169.111688,100.783310,470.408033
12,Belgium,BEL,2.2964,25.57,152.95,5.480131,8.980837,21.431876
15,Bulgaria,BGR,2.8096,40.11,31.05,33.027504,7.004737,82.342319
18,Croatia,HRV,0.4960,10.39,27.60,6.559420,4.773821,63.132053


In [258]:
styles.eco_colours

{'red': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4'}

In [ ]:
df_temp2 = gas_df[~gas_df['country'].isin(['Ireland'])].copy()

bars = alt.Chart(df_temp2).mark_bar(
    color = alt.expr("datum.country == 'United Kingdom' ? '#e6224b' : (datum.country == 'EU' ? '#eb5c2e' : '#122b39')")
).encode(
    alt.Y('country:N', sort='-x'),
    alt.X('capacity_load:Q').title('Gas storage (% of total capacity)').axis(
        labelExpr='datum.value > 0 ? datum.value + "%" : datum.value',
        labelAlign='center',
        zindex=1,
        titleFontSize=13
    )
)
labels = bars.mark_text(
    align=alt.expr('datum.capacity_load > 8 ? (datum.iso3 == "GBR" ? "left" : "right") : "left"'),
    color=alt.expr('datum.capacity_load > 8 ? (datum.iso3 == "GBR" ? "#e6224b": "white") : datum.iso3 == "GBR" ? "#e6224b" : "#122b39"'),
    dx=alt.expr('datum.capacity_load > 8 ? (datum.iso3 == "GBR" ? 3 : -3) : 4'),
    baseline='middle',
    # text = alt.expr('datum.storage_days > 0 ? format(datum.storage_days, ".1f") + " TWh" : "None"')
    text = alt.expr('datum.capacity_load == 0 ? "None" : datum.capacity_load < 0.1 ? "<0.1" : datum.iso3 == "GBR" ? format(datum.capacity_load, "d") + "%" : format(datum.capacity_load, "d")')
)


logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(0),url='img:N')


chart = (bars + labels + logo).properties(
    width=330,
    height=400
)
chart.title = alt.Title(
    ["Source: AGSI (European stock); National Gas (UK); Energy Institute (consumption)", "Notes: Stocks as of 7-9th March 2026; consumption based on 2024 figure"],
    orient='bottom', fontStyle='italic', fontSize=10, color='#676A8680', fontWeight='normal', frame='group', dx=0, offset=7
)

styles.save(chart, folder_path, f'{file_base}C2_gas_storage_capacity', width=330, height=400)

chart.display()

alt.LayerChart(...)

**Storage stock** days consumption

In [276]:
df_temp3 = gas_df[~gas_df['country'].isin(['Ireland'])].copy()

bars = alt.Chart(df_temp3).mark_bar(
    # color = alt.expr("datum.country == 'United Kingdom' ? '#e6224b' : '#122b39'")
    color = alt.expr("datum.country == 'United Kingdom' ? '#e6224b' : (datum.country == 'EU' ? '#eb5c2e' : '#122b39')")
).encode(
    alt.Y('country:N', sort='-x'),
    alt.X('storage_days:Q').title('Gas storage (days of consumption)').axis(
        labelExpr='datum.value > 0 ? datum.value + " days" : datum.value',
        labelAlign='center',
        zindex=1,
        titleFontSize=13
    )
)
labels = bars.mark_text(
    align=alt.expr('datum.storage_days > 10 ? "right" : "left"'),
    color=alt.expr('datum.storage_days > 10 ? "white" : datum.iso3 == "GBR" ? "#e6224b" : "#122b39"'),
    dx=alt.expr('datum.storage_days > 10 ? -4 : 4'),
    baseline='middle',
    # text = alt.expr('datum.storage_days > 0 ? format(datum.storage_days, ".1f") + " TWh" : "None"')
    text = alt.expr('datum.storage_days == 0 ? "None" : datum.storage_days < 0.1 ? "<0.1" : datum.iso3 == "GBR" ? format(datum.storage_days, "d") + " days" : format(datum.storage_days, "d")')
)


logo=alt.Chart(pd.DataFrame([{ "img": "https://raw.githubusercontent.com/EconomicsObservatory/ECOvisualisations/main/guidelines/logos/eco-icon-dark_cropped.png"}]))\
    .mark_image(width=40, height=20, align='right',baseline='bottom',opacity=mo,xOffset=0).encode(x=alt.value('width'),y=alt.value(0),url='img:N')


chart = (bars + labels + logo).properties(
    width=330,
    height=400
)

chart.title = alt.Title(
    ["Source: AGSI (European stock); National Gas (UK); as of 7-9th March 2026"],
    orient='bottom', fontStyle='italic', fontSize=10, color='#676A8680', fontWeight='normal', frame='group', dx=0, offset=7
)

chart.display()

alt.LayerChart(...)

In [ ]:
# styles.save(chart, pathlib.Path(''), '20260309_C3_gas_storage_days', width=330, height=400)
styles.save(chart, folder_path, f'{file_base}C3_gas_storage_days', width=330, height=400)

In [ ]:
gas_storage_raw = pd.read_csv('AGSI_gasDayStart_2026-03-03.csv', delimiter=';')
gas_storage_raw.head(1)

7th March
- 6.9998

In [280]:
# # Augment UK values 3rd March
# uk_data = {
#     'country': 'UK',
#     'storage_stock': 6.4639, 
#     'full_percent': 6.4639 / 35.415 * 100,
#     'consumption': 714.14,
# }
# eu_data = {
#     'country': 'EU',
#     'storage_stock': 339.9971,
#     'full_percent': 29.76,
#     'consumption': 3519.0,
# }

# Augment UK values
uk_data = {
    'country': 'UK',
    'storage_stock': 6.9998, 
    'full_percent': 6.9998 / 35.415 * 100,
    'consumption': 714.14,
}
eu_data = {
    'country': 'EU',
    'storage_stock': 335.9200,
    'full_percent': 29.40,
    'consumption': 3519.0,
}

eu_uk = pd.DataFrame([eu_data, uk_data])
eu_uk['storage_days'] = eu_uk['storage_stock'] / (eu_uk['consumption'] / 365)
# Calculate measure for size of maximum possible storage relative to consumption, as a measure of "capacity proficiency"
eu_uk['storage_maximum'] = eu_uk['storage_stock'] * 100 / eu_uk['full_percent']
eu_uk['capacity_proficiency'] = eu_uk['storage_maximum'] / (eu_uk['consumption'] / 365)
eu_uk = eu_uk.melt(id_vars=['country',], var_name='metric', value_name='value')
eu_uk

,country,metric,value
0,EU,storage_stock,335.920000
1,UK,storage_stock,6.999800
2,EU,full_percent,29.400000
3,UK,full_percent,19.765071
4,EU,consumption,3519.000000
5,UK,consumption,714.140000
6,EU,storage_days,34.842512
7,UK,storage_days,3.577628
8,EU,storage_maximum,1142.585034
9,UK,storage_maximum,35.415000


In [281]:
styles.eco_colours

{'red': '#e6224b',
 'blue-light': '#179fdb',
 'blue-dark': '#122b39',
 'yellow': '#f4c245',
 'orange': '#eb5c2e',
 'turquoise': '#36b7b4'}

In [ ]:
panel_width = 125

# Panel 1: Storage stock, % of capacity
p1 = alt.Chart(eu_uk).transform_filter(
    alt.datum.metric == 'full_percent'
).mark_bar().encode(
    alt.X('country:N'),
    alt.Y('value:Q').title(['Current gas storage,', '% of total capacity']).axis(titleY=-20, labelExpr='datum.value + "%"').scale(domain=[0, 40])
).properties(width=panel_width)
p1 += p1.mark_text(
    align='center', dx=0, dy=9, fontSize=13, color='white',
    text=alt.expr("format(datum.value, '.1f') + '%'")
)

# Panel 2: Current storage in days of consumption
p2 = alt.Chart(eu_uk).transform_filter(
    alt.datum.metric == 'storage_days'
).mark_bar().encode(
    alt.X('country:N'),
    alt.Y('value:Q').title(['Current gas storage,', 'days of consumption']).axis(
        titleY=-20,
        labelExpr='datum.value + " days"'
    ).scale(domain=[0, 45])
).properties(width=panel_width)
p2 += p2.mark_text(
    align='center', dx=0, dy=9, fontSize=13, color='white',
    text=alt.expr("format(datum.value, 'd')")
)

# Panel 3: Max capacity relative to annual consumption
p3 = alt.Chart(eu_uk).transform_filter(
    alt.datum.metric == 'capacity_proficiency'
).mark_bar().encode(
    alt.X('country:N'),
    alt.Y('value:Q').title(['Days consumption at', 'max storage capacity']).axis(
        titleY=-20,
        labelExpr='datum.value + " days"'
    ).scale(domain=[0, 135])
).properties(width=panel_width)
p3 += p3.mark_text(
    align='center', dx=0, dy=9, fontSize=13, color='white',
    text=alt.expr("format(datum.value, 'd')")
)

chart = alt.hconcat(p1, p2, p3).configure_axisX(labelFontSize=13)
chart.title = alt.Title(
    ["Source: AGSI (European stock); National Gas (UK); Energy Institute (consumption)", "Notes: Stocks as of 7-9th March 2026; consumption based on annual 2024 figure"],
    orient='bottom', fontStyle='italic', fontSize=10, color='#676A8680', fontWeight='normal', frame='group', dx=0, offset=7
)
chart.display()

# styles.save(chart, pathlib.Path(''), '20260309_C4_gas_eu_uk_comparison', width=None, height=None)
styles.save(chart, folder_path, f'{file_base}C4_gas_eu_uk_comparison', width=None, height=None)

alt.HConcatChart(...)

In [210]:
panel_width = 340
pheight = 100

# Panel 1: Storage stock, % of capacity
p1 = alt.Chart(eu_uk).transform_filter(
    alt.datum.metric == 'full_percent'
).mark_bar().encode(
    alt.Y('country:N'),
    alt.X('value:Q').axis(titleY=-20, labelExpr='datum.value + "%"').scale(domain=[0, 40])
).properties(width=panel_width, height=pheight)
p1 += p1.mark_text(
    align='right', dx=-3, dy=0, fontSize=13, color='white',
    text=alt.expr("format(datum.value, '.1f') + '%'")
)
p1.title = alt.Title('Current gas storage, % of total capacity', fontSize=13, anchor='middle', dx=0)

# Panel 2: Current storage in days of consumption
p2 = alt.Chart(eu_uk).transform_filter(
    alt.datum.metric == 'storage_days'
).mark_bar().encode(
    alt.Y('country:N'),
    alt.X('value:Q').axis(
        titleY=-20,
        labelExpr='datum.value + " days"'
    ).scale(domain=[0, 45])
).properties(width=panel_width, height=pheight)
p2 += p2.mark_text(
    align='right', dx=-3, dy=0, fontSize=13, color='white',
    text=alt.expr("format(datum.value, 'd')")
)
p2.title = alt.Title('Current gas storage, days of consumption', fontSize=13, anchor='middle', dx=0)

# Panel 3: Max capacity relative to annual consumption
p3 = alt.Chart(eu_uk).transform_filter(
    alt.datum.metric == 'capacity_proficiency'
).mark_bar().encode(
    alt.Y('country:N'),
    alt.X('value:Q').title('').axis(
        titleY=-20,
        labelExpr='datum.value + " days"'
    ).scale(domain=[0, 135])
).properties(width=panel_width, height=pheight)
p3 += p3.mark_text(
    align='right', dx=-3, dy=0, fontSize=13, color='white',
    text=alt.expr("format(datum.value, 'd')")
)
p3.title = alt.Title('Max storage capacity (days of consumption)', fontSize=13, anchor='middle', dx=0)

alt.vconcat(p1, p2, p3).configure_axis(labelFontSize=13)

alt.VConcatChart(...)

Chart thread:
1. Gas storage stock (bar chart, UK & 19 European countries)
2. Gas capacity load (bar chart, UK & 19 European countries)
3. Gas stocks relative to consumption, in days (bar chart, UK & 19 European countries)
4. UK vs Europe 
    - idea: not really relevant to directly compare to all European countries, due to EU gas sharing
    - Explanation fo European sharing here: https://www.consilium.europa.eu/en/infographics/gas-storage-capacity/
    - Three panel chart comparing UK vs EU for current storage stock, storage stock days of consumption, days consumption at max storage (i.e. proficiency of max storage capacity). 

In [ ]:
# # Fetch natural gas consumption
# iea_data = pd.read_excel('IEA-World Energy Balances Highlights 2025.xlsx', sheet_name='TimeSeries_1971-2024', skiprows=1)

# iea_data['ISO3'] = coco.convert(iea_data['Country'], to='ISO3')

# print('Products: ', iea_data['Product'].unique())
# print('Flows: ', iea_data['Flow'].unique())

# # Filter to natural gas consumption
# gas = iea_data[(iea_data['Product'] == 'Natural gas') & (iea_data['Flow'].str.contains('Total final consumption', case=False))].copy()

# gas = gas[['ISO3', 'Country', '2023']].rename(columns={'2023': 'consumption (PJ)'})
# #
# gas['consumption (TWh)'] = gas['consumption (PJ)'] / (1/0.2778)  # Convert PJ to TWh
# gas.sample(5)

# #  Not all gas consumption is 'final consumption', so need to either find other, or use IEA 'share of has used by final consumers' to transform given final consumption to actual total. https://www.iea.org/countries/france/natural-gas

# # Inline gas storage data
# storage = pd.DataFrame({
#     'ISO3': ['GBR', 'DEU', 'FRA', 'ITA', 'ESP', 'NLD'],
#     'Country': ['United Kingdom', 'Germany', 'France', 'Italy', 'Spain', 'Netherlands'],
#     'storage': [2.8966, 52.3011, 27.2224, 96.7929, 20.8014, 15.2278],  # Live storage values in TWh
#     'percent_full': [29.37, 20.83, 21.65, 47.60, 58.05, 10.55],  # % of storage capacity filled
#     'gas_share_final': [60, 75, 80, 59, 53, 65] # Share of gas used by final consumers 
# })

# df = storage.merge(gas, on=['ISO3', 'Country'], how='inner', suffixes=('_storage', '_consumption'))

# df['consumption_total'] = df['consumption (TWh)'] / (df['gas_share_final'] / 100)  # Calculate total consumption based on gas share

# df['supply_days'] = df['storage'] / (df['consumption_total'] / 365)  # Calculate supply days
# df

OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total not found in regex
OECD Total